# Notebook 12 — Computability and Decidability

**Covers Chapter 5 §5.1–5.3.** From "how to prove a program correct" (chapter 4) to **"can this even be computed?"** — a deeper question that turns out to have a *no* answer for some functions, and a tantalising *maybe* for others.

This is the most theoretical chapter so far. Three big ideas:

1. **Definition of computable functions** — formalised via Hoare triples.
2. **Uncomputable functions exist** — pure cardinality argument: there are *more* functions than there are programs.
3. **Decidable vs semidecidable predicates** — for yes/no problems, two levels of "answerable."

## What you'll be able to do by the end

- State what "f is computable" means in our setting (with reference to Hoare triples).
- Prove a specific function is computable by exhibiting a witness program.
- Explain *why* some functions can't be computed without writing down a specific example.
- Distinguish decidable, semidecidable, and undecidable predicates.
- Solve exercises 39–50.

In [1]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks']:
    if (candidate / 'while_lang.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise ImportError('could not find while_lang.py')

from while_lang import (
    parse, run, trace, count_steps,
    verify_triple, report_triple,
    StepBudgetExceeded,
)
import math
print('imports OK')

imports OK


## §1. Computable functions — Definition 14

We want to give a precise meaning to **"the function f can be computed."** The chapter does this by piggy-backing on Hoare logic from chapter 4:

$$f : \mathbb{N} \to \mathbb{N} \text{ is **computable** iff there exists a While program S such that}\quad \{n = \bar n \in \mathbb{N}\}\ S\ \{\Downarrow x = f(\bar n)\}$$

Reading: there's a program that takes any natural number `n` and *terminates* with `f(n)` in the output variable `x`.

**Key bits:**

- The triple uses **total correctness** (`⇓`) — the program must always terminate for natural-number inputs.
- Input lives in `n`, output in `x`. (Convention.)
- The program *witnesses* the computability — having one is enough, you don't need to find the most efficient or shortest.

### Three quick examples

**Example 12 — identity.** `id(n) = n`. Program: `x := n`. Trivially total. ✓

**Example 13 — add a constant.** `f(n) = n + m` (for some fixed `m`). Program: `x := n + m`. ✓

**Example 14 — `n mod 7`.** Program:
```
x := n;
while 7 ≤ x do x := x − 7
```
Loop terminates (variant: `x`, invariant: `x ≥ 0 ∧ x ≡ n̄ (mod 7)`). ✓

In [2]:
# Verify Examples 12-14 empirically.
id_prog       = 'x := n'
add5_prog     = 'x := n + 5'
mod7_prog     = '''
    x := n;
    while 7 <= x do x := x - 7
'''

# Each computable function = (witness program, expected output for input n)
examples = [
    ('identity',       id_prog,    lambda n: n),
    ('add 5',          add5_prog,  lambda n: n + 5),
    ('mod 7',          mod7_prog,  lambda n: n % 7),
]

for label, prog, expected in examples:
    ok = 0
    for n in range(0, 30):
        # Need brackets around while body for our parser
        p = prog.replace('do x := x - 7', 'do (x := x - 7)')
        res = verify_triple(
            precond=lambda s, n=n: s.get('n', 0) == n,
            prog=p,
            postcond=lambda s, n=n, e=expected: s.get('x', 0) == e(n),
            sample_states=[{'n': n}],
            mode='total',
        )
        ok += res['verified']
    print(f'{label:12}: verified for {ok}/30 natural-number inputs')

identity    : verified for 30/30 natural-number inputs
add 5       : verified for 30/30 natural-number inputs
mod 7       : verified for 30/30 natural-number inputs


## §2. Computability for partial functions — Definition 15

Some functions aren't defined everywhere. The integer-square-root function `√` is only defined when its input is a perfect square (in ℕ); the equation-solver from §4.7.3 is only defined when an integer solution exists.

A **partial** function `f : ℕ ⇀ ℕ` (notation: half-arrow) maps each input to *at most* one output. Computability for partials:

$$f \text{ is computable iff there exists S such that}\quad \{n = \bar n \in \mathbb{N} \,\land\, f(\bar n) \text{ defined}\}\ S\ \{\Downarrow x = f(\bar n)\}$$

**Key permission:** if `f(n̄)` is *not* defined, the program is allowed to do anything — return junk, never terminate, whatever. We just don't make a claim.

**Key consistency:** if `f` happens to be total, this definition collapses to Definition 14.

**Example 15 — `n²` for even n.**
```
x := n;
while 2 ≤ n do (n := n − 2);
if n = 0 then x := x × x else skip
```
Computes `n²` if `n` is even. For odd `n`, it returns `n` (which isn't `n²`) — but the spec doesn't claim anything for odd inputs, so it's still a witness.

## §3. Generalising — Definition 16

What about functions like `f : ℤ → ℕ` (input is signed) or `f : ℕ × ℕ → ℕ` (input is a pair)? While only handles natural-number variables, but **we can encode** other sets via injections.

**Definition 16** (paraphrased): given sets `S, T` and injections `γ : ℕ → S`, `δ : T → ℕ`, a partial function `f : S ⇀ T` is **computable** iff the composite

$$\mathbb{N} \xrightarrow{\gamma} S \xrightarrow{f} T \xrightarrow{\delta} \mathbb{N}$$

is computable in the sense of Definition 15.

**In words:** if you can encode S and T as natural numbers (via γ and δ), then "f is computable" reduces to the encoded version being computable.

**Why this matters:** we don't need separate notions of computability for ℤ-valued, pair-valued, or boolean-valued functions. They all reduce to ℕ → ℕ via encoding. **The choice of encoding doesn't affect *whether* a function is computable** — provided the encoding itself is computable (which it always is for the standard cases).

Example uses:
- For `f : ℕ → 𝔹`, use `δ(tt) = 1, δ(ff) = 0` (this becomes the *characteristic function* of a predicate — see §5).
- For `f : ℤ → ℕ`, use `γ = β` (Section 6.2.1's bijection) — turn ℤ inputs into ℕ inputs.
- For `f : ℕ × ℕ → ℕ`, use `γ = φ` (Section 6.2.2's pairing) — turn pair inputs into ℕ inputs.

## §4. There are uncomputable functions — Proposition 17

**The big result.** Not every function `ℕ → ℕ` is computable. The proof is a **cardinality** argument — we don't exhibit a specific uncomputable function, we just show there must be one because the sets have different sizes.

### Two facts from Section 6.3 (taken on faith here)

1. **The set of While programs is countable.** Every program is a finite string over a finite alphabet. The set of finite strings over a finite alphabet is countable.
2. **The set `Fun(ℕ, ℕ)` is uncountable.** Cantor's diagonal argument: given any putative enumeration `f₁, f₂, f₃, …` of functions, define `g(n) = f_n(n) + 1`. Then `g ≠ f_n` for any `n`. So the enumeration was incomplete — `Fun(ℕ, ℕ)` can't be put in a list.

### The argument

- Each computable function has at least one witness program. So the set of computable functions has cardinality ≤ |While programs| = ℵ₀ (countable infinity).
- `Fun(ℕ, ℕ)` has cardinality 2^ℵ₀ (uncountable infinity, by Cantor).
- 2^ℵ₀ > ℵ₀, so there are *strictly more* functions than computable ones.

**Conclusion:** there exist functions `ℕ → ℕ` that are not computable. ∎

### What this argument doesn't tell us

- It doesn't give an example.
- It doesn't say *most* functions are uncomputable (though they are — "most" of an uncountable set is uncountable).
- It doesn't constrain *which* functions are uncomputable.

For an actual example we have to wait until §5.4 — the **Halting Problem** is a concrete uncomputable predicate.

## §5. Decidability — Definition 18

Many natural questions are yes/no rather than "compute this number." Three equivalent ways to think about a yes/no problem on naturals:

1. **A predicate `P` on ℕ.** `P(n)` either holds or doesn't.
2. **A function `f : ℕ → 𝔹`.** Returns `tt` if yes, `ff` if no.
3. **A subset `A ⊆ ℕ`.** The set of inputs for which the answer is yes.

Definition 18: `P` is **decidable** iff its **characteristic function**

$$\chi_P(x) = \begin{cases} \mathbf{tt} & P(x) \\ \mathbf{ff} & \text{else}\end{cases}$$

is computable (via Definition 16, encoding `tt ↦ 1, ff ↦ 0`).

**Equivalently:** there's a While program S with

$$\{n = \bar n \in \mathbb{N}\}\ S\ \{\Downarrow (P(\bar n) \to x = 1) \,\land\, (\neg P(\bar n) \to x = 0)\}$$

Such an S is called a **decision procedure** for `P`.

**Example 16 — "is n the i-th triangular number for some i?"**

```
z := 1; y := 1;
while z < n do (y := y + 1; z := z + y);
if z = n then x := 1 else x := 0
```

The `while` builds up triangular numbers `T(1), T(2), T(3), …` until reaching or exceeding `n`. Then the `if` checks whether we landed exactly on `n`. **Always terminates** (variant: `n − z` while `z < n`). Computes `χ_P`. ✓

In [3]:
# Empirically check Example 16 — triangular-number predicate.
tri_prog = '''
    z := 1; y := 1;
    while z <= n - 1 do (y := y + 1; z := z + y);
    if z = n then x := 1 else (x := 0)
'''
# Note: "z < n" written as "z <= n - 1" since While has only <=.

# Triangular numbers: 1, 3, 6, 10, 15, 21, 28, ...
triangulars = {1, 3, 6, 10, 15, 21, 28, 36, 45, 55}

ok = 0
for n in range(1, 60):
    expected = 1 if n in triangulars else 0
    res = verify_triple(
        precond=lambda s, n=n: s.get('n', 0) == n,
        prog=tri_prog,
        postcond=lambda s, e=expected: s.get('x', 0) == e,
        sample_states=[{'n': n}],
        mode='total',
    )
    ok += res['verified']

print(f'triangular-number predicate decidable: {ok}/59 verified')

triangular-number predicate decidable: 59/59 verified


## §6. Decidability is closed under logic — Proposition 19

Given decidable predicates `P, Q`, all four of these are decidable:

1. **`¬P`** — characteristic function: `1 − χ_P(x)`.
2. **`P ∧ Q`** — characteristic function: `χ_P(x) · χ_Q(x)`.
3. **`P ∨ Q`** — characteristic function: max of the two (or `χ_P + χ_Q − χ_P·χ_Q`).
4. **`P → Q`** — characteristic function: `(1 − χ_P) ∨ χ_Q`.

**Proof shape (for ¬P):**

Given a decision procedure `S` for `P`, build a new procedure `T`:
```
S;
if x = 0 then x := 1 else x := 0
```

This flips `S`'s answer. Then `T` computes `χ_¬P` — by total-correctness derivation: `S` ends with `x = χ_P(n̄)`, the if flips it to `x = 1 − χ_P(n̄) = χ_¬P(n̄)`.

**The lesson:** decidability is closed under all the logical operators. So once you have a few core decidable predicates, you can build up complex ones.

## §7. Semidecidable predicates — Definition 20

What about predicates we *can't* decide but where we can at least *recognise* yes-instances?

**Definition 20:** `P` is **semidecidable** iff the partial function

$$g(x) = \begin{cases} \mathbf{tt} & P(x) \\ \text{undefined} & \text{else}\end{cases}$$

is computable. Equivalently: there's a While program S with

$$\{n = \bar n \in \mathbb{N} \,\land\, P(\bar n)\}\ S\ \{\Downarrow x = 1\}$$

**Reading:** if `P(n̄)` holds, S terminates with `1`. If `P(n̄)` doesn't hold, S can do *anything* (loop forever, return garbage, …) — we make no claim.

**Why this is interesting.** Some problems we can't decide can still be semidecided — we can recognise solutions when they exist, we just can't know when one *doesn't* exist.

### Famous semidecidable-but-not-decidable problems

- **The Halting Problem** (chapter §5.4 / our N13) — semidecidable. We can run a program and see if it stops; we can't (in general) tell when it won't stop.
- **First-order logic validity** — semidecidable (theorem provers). Undecidable (Church's theorem).
- **Diophantine equations** — Hilbert's tenth problem. Undecidable in general.

**Hierarchy:** undecidable ⊋ semidecidable ⊋ decidable. Each containment is strict.

### Trivially: decidable ⟹ semidecidable

If you have a decision procedure `S` for `P`, build a semi-decider `T`:
```
S;
if x = 1 then skip else (while tt do skip)
```
If `P(n̄)`: `S` returns 1, then if-true branch fires, T terminates with `x = 1`.  
If `¬P(n̄)`: `S` returns 0, then if-false branch fires the infinite loop. T diverges — fine for the partial spec.

**Or even simpler:** just run S. The total guarantee is *stronger* than the partial guarantee, so any decider trivially is a semi-decider for the same predicate. (The notebook reasoning above shows that even with an explicit non-terminating program, semidecidability still falls out.)

## §8. Exercises 39–43 — proving things are computable

## Exercise 39

> Show the following are computable.

### (a) `n ↦ n²`

Witness program: `x := n × n`.

Triple: `{n = n̄ ∈ ℕ} x := n × n {⇓ x = n̄²}` — by `:=` rule directly. ✓

### (b) Composite `g ∘ f`

Given computable `f : ℕ → ℕ` and `g : ℕ → ℕ` with witness programs `S_f` (input `n`, output `x`) and `S_g` (input `n`, output `x`).

**Witness program for `g ∘ f`:**
```
S_f;     // computes f(n̄) into x
n := x;  // pipe x to n
S_g      // computes g(f(n̄)) into x
```

**Triple to derive:**

$$\{n = \bar n \in \mathbb{N}\}\ S_f;\, n := x;\, S_g\ \{\Downarrow x = g(f(\bar n))\}$$

**Derivation:**
```
1. {n = n̄ ∈ ℕ} S_f {⇓ x = f(n̄)}                            given (S_f computes f).
2. {x = f(n̄)} n := x {⇓ n = f(n̄) ∧ n ∈ ℕ}                    by := rule.
3. {n = f(n̄) ∈ ℕ} S_g {⇓ x = g(f(n̄))}                       by computability of g (with the input being f(n̄)).
4. {n = n̄ ∈ ℕ} S_f; n := x; S_g {⇓ x = g(f(n̄))}              by ; rule applied to 1, 2, 3.
```

### (c) Even-ness composed with a computable function

Given computable `f`, define `g(n) = 1 if f(n) is even else 0`. Witness program (assuming we have a program `MOD2` computing `n mod 2` into `x`):
```
S_f;     // computes f(n̄) into x
n := x;  // pipe to n
MOD2;    // computes (f(n̄)) mod 2 into x
if x = 0 then x := 1 else x := 0
```

Triple: `{n = n̄ ∈ ℕ} S {⇓ (f(n̄) even → x = 1) ∧ (f(n̄) odd → x = 0)}`.

Derivation: chain three `;` applications and an `if` rule, plus consequence to clean up the post.

In [4]:
# Verify (a) n^2 computability empirically.
sq_prog = 'x := n * n'
ok = 0
for n in range(0, 30):
    res = verify_triple(
        precond=lambda s, n=n: s.get('n', 0) == n,
        prog=sq_prog,
        postcond=lambda s, n=n: s.get('x', 0) == n*n,
        sample_states=[{'n': n}],
        mode='total',
    )
    ok += res['verified']
print(f'n² computable: verified for {ok}/30 inputs')

# Verify (c) by composing 'square' and 'is-even' on f(n) = 2n + 1 (always odd → output should always be 0).
even_after_2nplus1 = '''
    x := 2 * n + 1;
    n := x;
    while 2 <= n do (n := n - 2);
    if n = 0 then x := 1 else (x := 0)
'''
ok = 0
for n in range(0, 20):
    expected = 0      # 2n+1 is always odd
    res = verify_triple(
        precond=lambda s, n=n: s.get('n', 0) == n,
        prog=even_after_2nplus1,
        postcond=lambda s, e=expected: s.get('x', 0) == e,
        sample_states=[{'n': n}],
        mode='total',
    )
    ok += res['verified']
print(f'(c) is-even-after-(2n+1) computable: verified for {ok}/20')

n² computable: verified for 30/30 inputs
(c) is-even-after-(2n+1) computable: verified for 20/20


## Exercise 40

### (a) `n ↦ 1 if n prime else 0`

We've already built this in N2 (Example 4). Witness program:
```
y := n - 1; z := 1;
while 2 ≤ y ∧ z = 1 do (
    r := n;
    while y ≤ r do r := r - y;
    if r = 0 then z := 0 else skip;
    y := y - 1
)
x := z
```
Total correctness Hoare triple: `{n = n̄ ∈ ℕ ∧ n̄ ≥ 2} S {⇓ x = 1 ↔ n̄ prime}`. Termination by the variant `y` (decreases each outer iteration). For `n̄ < 2`, primality isn't defined; flag as a precondition restriction.

### (b) β and β' from §6.2.1 — adjust the definition

These are functions ℤ → ℕ (β) and ℕ → ℤ (β'), so they don't fit Definition 14 directly. Use Definition 16:

- For β: `γ = β` itself (used as the encoding from ℕ to ℤ — wait, β goes ℤ → ℕ, so we need its inverse as the input encoding). More precisely, we treat β as the *encoding* between ℕ and ℤ, then any function involving ℤ becomes one on ℕ via β (in either direction).

**Easier approach:** β is the program in §4.7.1 — `if 0 ≤ n then x := 2 × n else x := -2 × n − 1`. We've shown this is total-correct. Done.

### (c) φ and φ⁻¹ — pairing and unpairing

Same idea: φ encodes ℕ × ℕ as ℕ; φ⁻¹ decodes. Standard candidate: Cantor's pairing function. **Informally computable** by a program that computes `(m+n)(m+n+1)/2 + n` for φ, and inverts via solving for the diagonal. Without §6.2.2's exact definition, we can't formalise; my answer takes the standard form and notes verification with a GTA.

## Exercise 41 — partial functions

> Show the following partial functions are computable.

### (a) `x ↦ x/2 if x/2 ∈ ℤ else undefined`

Witness:
```
x := n;
while 2 ≤ x do x := x − 2
// after loop, x ∈ {0, 1}
if x = 0 then
    // n was even — recompute n/2
    x := 0; y := n;
    while 2 ≤ y do (x := x + 1; y := y - 2)
else
    while tt do skip   // diverge for odd n
```
Spec: `{n = n̄ ∈ ℕ ∧ n̄ even} S {⇓ x = n̄/2}`. For odd `n̄`, no claim.

### (b) `x ↦ √x if √x ∈ ℕ else undefined`

Witness program: try increasing values of `z` and check `z² = n`. Diverge if no match found:
```
z := 0;
while ¬(z × z = n) ∧ z × z ≤ n do z := z + 1;
if z × z = n then x := z else (while tt do skip)
```
Spec: `{n = n̄ ∈ ℕ ∧ n̄ is a perfect square} S {⇓ x = √n̄}`. For non-squares we'd need the inner loop to diverge — but the search-up-to-n version above terminates and then the if branch handles the case.

Actually with the integer-sqrt program from chapter 1's Example 5 we already have a *total* function (it returns `⌊√n⌋`), which is stronger than the partial version we need.

### (c) `x ↦ 1 if ax² + bx + c = 0 else undefined`

If the equation has *no* solution at `x = n̄`, undefined. The witness:
```
if a × n × n + b × n + c = 0 then
    x := 1
else
    while tt do skip
```
Spec: `{n = n̄ ∈ ℕ ∧ a·n̄² + b·n̄ + c = 0} S {⇓ x = 1}`. ✓

### (d) `(m, n) ↦ 1 if ∃x, y. mx + ny = k`, else undefined — adjust definition

Use Definition 16 with `γ = φ⁻¹` to decode the input. The witness is the Diophantine searcher from Exercise 7 (returns 1 if a solution exists; loops forever otherwise):
```
// Decode pair (m̄, n̄) from input n
// then search for x, y satisfying mx + ny = k
// (program from Ex 7, with z := 1 added at the start of the success branch)
```

## Exercise 42 — using Definition 16 for β, β', φ, φ⁻¹

The point of this exercise: **does using the generalized definition matter** vs. the ad-hoc adjustments in Ex 39/40?

**Answer: no, for these functions.** Definition 16 says `f : S ⇀ T` is computable iff `δ ∘ f ∘ γ` is computable as a partial ℕ ⇀ ℕ function.

For β: `S = ℤ, T = ℕ`. The natural encoding `γ = β` itself (composes to identity for the relevant direction). The proof reduces to showing the chapter's β-program from §4.7.1 is total-correct, which we did in N9 Ex 31.

For β', φ, φ⁻¹: same pattern. The choice of encoding matters only insofar as the encoding itself is computable — for the standard ones (β, Cantor pairing), they are.

**Lesson:** Definition 16 is *not* fundamentally more powerful — it's a *systematic way to talk about* computability across encodings, so we don't need ad-hoc adjustments per source/target type.

## Exercise 43 — generalised computability for ℕ × ℕ

### (a) Swap function `ℕ × ℕ → ℕ × ℕ, (x, y) ↦ (y, x)`

**Direct definition:** "the swap function is computable iff there's a program that takes encoded `(x̄, ȳ)` in some input variable and writes encoded `(ȳ, x̄)` to the output."

**Definition 16 version:** for `S = ℕ × ℕ` and `T = ℕ × ℕ`, both with the standard pairing encoding (Cantor `φ`):

$$\delta \circ \text{swap} \circ \gamma = \varphi \circ \text{swap} \circ \varphi^{-1}$$

is the function `n ↦ φ(swap(φ⁻¹(n)))`. Witness program: decode `n` into `(m̄, n̄)` via `φ⁻¹`, then re-encode `(n̄, m̄)` via `φ`.

### (b) Subtraction `ℕ × ℕ → ℤ, (x, y) ↦ x − y`

Output is in ℤ, so use `δ = β` (the ℤ → ℕ encoding from §6.2.1).

**Witness:** decode input pair, compute the (signed) difference, encode result via β.

Both programs combine the encoding/decoding programs (Ex 8/9 in N2) with a pure-ℕ computation in the middle. **The composition of computable programs is computable** — this is a recurring theme.

## Exercise 44 — three decidable predicates

### (a) Evenness on ℕ

Decision procedure (we wrote this in N6 Ex 5):
```
is_odd := 0;
m := n;
while 1 ≤ m do (m := m − 1; is_odd := 1 − is_odd);
if is_odd = 0 then x := 1 else x := 0
```
Always terminates (variant `m`); returns 1 iff `n̄` is even. ✓

### (b) Primality on ℕ

Decision procedure: the chapter's Example 4 (we used it in N2 Ex 4 and N6 Ex 20). Always terminates (nested variants); returns 1 iff `n̄` is prime. ✓

### (c) "Last digit of n is divisible by 3"

Decision procedure: from N10 Ex 37. Compute `n mod 10`, then check that mod 3 = 0. Always terminates; returns 1 iff the last decimal digit is in {0, 3, 6, 9}. ✓

## Exercise 45 — decidability of `P ∧ Q`

> Show that if P and Q are decidable, so is P ∧ Q.

**Construction.** Let `S_P` decide `P` and `S_Q` decide `Q`. Build the new program `T`:
```
S_P;
if x = 1 then
    S_Q
else
    skip
```

**Why it works (informally):**
- If `P(n̄)`: `S_P` returns 1, then we run `S_Q` which returns 1 iff `Q(n̄)`. So `T` returns 1 iff both hold.
- If `¬P(n̄)`: `S_P` returns 0, the if-false fires, the program ends with `x = 0`. So `T` returns 0 (matches `(P ∧ Q)(n̄)`'s value).

**Total correctness.** `T` is a sequence of total programs (`S_P` and `S_Q` are total by hypothesis; `skip` is trivially total). Hence `T` is total. ✓

**Hoare triple:** `{n = n̄ ∈ ℕ} T {⇓ (P(n̄) ∧ Q(n̄) → x = 1) ∧ (¬(P(n̄) ∧ Q(n̄)) → x = 0)}`.

Derivation chains the assumed triples for `S_P` and `S_Q` via `;` and applies the `if` rule. ∎

## Exercise 46 — decidability of `P ∨ Q` and `P → Q`

### `P ∨ Q`

```
S_P;
if x = 1 then
    skip   // already 1
else
    S_Q    // x becomes 1 iff Q(n̄)
```

If `P(n̄)`: returns 1. Else: defer to `S_Q`, which returns `χ_Q(n̄)`. So the result is `χ_P(n̄) ∨ χ_Q(n̄) = χ_{P∨Q}(n̄)`. ✓

### `P → Q`

Use `P → Q ≡ ¬P ∨ Q`:
```
S_P;
if x = 0 then
    x := 1   // ¬P holds, so P → Q is true
else
    S_Q       // P holds, so P → Q is just Q
```

Same kind of derivation as `P ∨ Q`. ✓

## Exercise 47 — total-correctness derivations for ¬P and P ∨ Q

### Derivation for `¬P`

Let `T = S_P; if x = 0 then x := 1 else x := 0`. Hypothesis: `S_P` is a decision procedure for `P`:

$$\{n = \bar n \in \mathbb{N}\}\ S_P\ \{\Downarrow (P(\bar n) \to x = 1) \,\land\, (\neg P(\bar n) \to x = 0)\}$$

**Derivation:**
```
1. {n = n̄ ∈ ℕ} S_P {⇓ (P(n̄) → x = 1) ∧ (¬P(n̄) → x = 0)}    by hypothesis.

2. {(P(n̄) → x = 1) ∧ (¬P(n̄) → x = 0) ∧ x = 0} x := 1
         {⇓ (¬P(n̄) → x = 1) ∧ (P(n̄) → x = 0)}                 by := rule.
         (Pre implies ¬P(n̄): if x = 0 and (P(n̄) → x = 1), 
         then ¬P(n̄). Hence post: x = 1 and ¬P(n̄), so the
         right post holds.)

3. {(P(n̄) → x = 1) ∧ (¬P(n̄) → x = 0) ∧ ¬(x = 0)} x := 0
         {⇓ (¬P(n̄) → x = 1) ∧ (P(n̄) → x = 0)}                 by := rule.
         (Symmetric: x ≠ 0 implies P(n̄); post becomes x = 0
         with P(n̄) holding.)

4. {(P(n̄) → x = 1) ∧ (¬P(n̄) → x = 0)}
         if x = 0 then x := 1 else x := 0
         {⇓ (¬P(n̄) → x = 1) ∧ (P(n̄) → x = 0)}                 from 2, 3 by if rule.

5. {n = n̄ ∈ ℕ} T {⇓ (¬P(n̄) → x = 1) ∧ (P(n̄) → x = 0)}        from 1, 4 by ; rule.
```

Step 5's postcondition is exactly `χ_¬P(n̄)`'s spec. ∎

### Derivation for `P ∨ Q`

Similar pattern with two conditional branches running `skip` vs `S_Q`. The if rule combines the two branches. The Q's spec gives us the missing piece in the else branch.

## Exercise 48 — decidable ⟹ semidecidable

**Claim:** if `P` is decidable, then `P` is semidecidable.

**Proof.** Let `S` be a decision procedure for `P` (always terminates, returns 1 iff `P` holds). We need a witness for semidecidability:

$$\{n = \bar n \in \mathbb{N} \,\land\, P(\bar n)\}\ T\ \{\Downarrow x = 1\}$$

Take `T = S`. By decidability:

$$\{n = \bar n \in \mathbb{N}\}\ S\ \{\Downarrow (P(\bar n) \to x = 1) \,\land\, (\neg P(\bar n) \to x = 0)\}$$

Strengthen the precondition to `n = n̄ ∈ ℕ ∧ P(n̄)`. Then `P(n̄) → x = 1` simplifies to `x = 1` (since `P(n̄)` is true). By rule of consequence: `{n = n̄ ∈ ℕ ∧ P(n̄)} S {⇓ x = 1}`. ✓

(Alternative phrasing: total-correctness is strictly stronger than the partial spec required for semidecidability — the inclusion is automatic.)

## Exercise 49 — semidecidability of `¬P` and `P ∧ Q`

### `P ∧ Q`: semidecidable if both `P, Q` are

Let `S_P` semi-decide `P`, `S_Q` semi-decide `Q`. Run them sequentially:
```
S_P;
S_Q
```

If both hold: `S_P` terminates with `x = 1`, then `S_Q` terminates with `x = 1`. ✓  
If `P(n̄)` fails: `S_P` may not terminate, so the program may not terminate. **No claim required for `¬P` case** — that's fine for semidecidability of `P ∧ Q`.  
If `Q(n̄)` fails (but `P` holds): `S_P` terminates, then `S_Q` may not terminate. Again no claim required. ✓

### `¬P`: NOT necessarily semidecidable

**Claim:** if `P` is semidecidable, `¬P` may NOT be.

**Reason:** if both `P` and `¬P` were semidecidable, then `P` would be decidable — run both semi-deciders "in parallel" (interleave steps); whichever terminates first tells you the answer. So if every semidecidable predicate had a semidecidable negation, every semidecidable predicate would be decidable.

**But:** the Halting Problem is semidecidable and *not* decidable (Theorem 21). So the negation of Halting (which is also undecidable but *not* even semidecidable — it's co-semidecidable) gives a counter-example.

**Verdict:** `¬P` is NOT semidecidable in general. A program witnessing it cannot exist.

## Exercise 50 — semidecidability of `P ∨ Q` and `P → Q`

### `P ∨ Q`: semidecidable if both `P, Q` are

We need a program that terminates with `x = 1` if either `P(n̄)` or `Q(n̄)` holds. **Trick: dovetailing** — interleave the steps of `S_P` and `S_Q`. As soon as either terminates, return `1`.

Pseudo-program (dovetailing isn't trivial in While, but conceptually):
```
for k = 0, 1, 2, ...:
    run S_P for k steps on input n̄;
    if it terminated with x = 1, return 1;
    run S_Q for k steps on input n̄;
    if it terminated with x = 1, return 1
```

If at least one of `P, Q` holds, eventually we find a k for which the corresponding `S` terminates. Witness program ✓.

### `P → Q`: NOT necessarily semidecidable

**Reason:** `P → Q ≡ ¬P ∨ Q`. By Ex 49, `¬P` need not be semidecidable. By the dovetailing argument, if both `¬P` and `Q` were semidecidable then `¬P ∨ Q` would be — but we can't always semidecide `¬P`.

**Counter-example:** take `Q` always-false (`Q(n) = ⊥` for all n). Then `P → Q ≡ ¬P`, so semidecidability of `P → Q` ≡ semidecidability of `¬P` — and we know `¬P` isn't semidecidable in general.

**Verdict:** `P → Q` is NOT semidecidable in general.

### Summary table for closure under logic

| Operation | Decidable? | Semidecidable? |
|:--|:-:|:-:|
| `P` (decidable) | ✓ | ✓ (Ex 48) |
| `¬P` | ✓ | ✗ in general (Ex 49) |
| `P ∧ Q` | ✓ | ✓ (Ex 49) |
| `P ∨ Q` | ✓ | ✓ (dovetailing, Ex 50) |
| `P → Q` | ✓ | ✗ in general (Ex 50) |

**The key insight:** decidable predicates are a Boolean algebra; semidecidable predicates are closed under `∧` and `∨` (positive logic) but **not** under `¬` or `→`.

## Summary

**Computable function** `f : ℕ → ℕ` = there's a witness While program S with `{n = n̄ ∈ ℕ} S {⇓ x = f(n̄)}`.

**Partial computable function** = same, but only required for inputs where `f` is defined.

**Definition 16** generalises to `f : S ⇀ T` via injections `γ : ℕ → S`, `δ : T → ℕ`. Reduces all computability questions to ℕ ⇀ ℕ.

**Uncomputable functions exist** (Proposition 17) — pure cardinality argument. `Fun(ℕ, ℕ)` is uncountable; programs are countable. Proves existence without giving an example.

**Decidability** for predicates: `P` is decidable iff `χ_P` (its characteristic function) is computable. Equivalently: there's a decision procedure that always terminates.

**Semidecidability:** weaker — only need to recognise yes-instances. `S` terminates with `1` if `P(n̄)`; can do anything if `¬P(n̄)`.

**Closure properties:**
- Decidable: closed under `¬, ∧, ∨, →` (all logical operators).
- Semidecidable: closed under `∧, ∨` only. Negation and implication can break it.

**Hierarchy:** decidable ⊊ semidecidable ⊊ all-predicates. Both inclusions strict.

**Next:** N13 covers the **Halting Problem** (an explicit semidecidable-but-not-decidable predicate), the universal program, and the Church-Turing thesis.